In [39]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import os

# --- Page Configuration ---
st.set_page_config(
    page_title="Placement Readiness & Job Predictor",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ... [PASTE ALL YOUR STREAMLIT UI CODE HERE] ...

"""
 Streamlit Web App - Placement Readiness & Job Role Predictor
 Course: Applied Machine Learning (22MET921)
 Instructor: Dr. Gunjan Soni
 Team: Swapnil Acharya, Aditya Verma, Amit Kumar
"""

import streamlit as st
import pandas as pd
import numpy as np
import pickle
import os

# ─── Page Configuration ───
st.set_page_config(
    page_title="Placement Readiness & Job Predictor",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ─── Custom CSS ───
st.markdown("""
<style>
    .main-header {
        font-size: 2.2rem;
        font-weight: bold;
        color: #1f4e79;
        text-align: center;
        margin-bottom: 0.5rem;
    }
    .sub-header {
        font-size: 1.1rem;
        color: #555;
        text-align: center;
        margin-bottom: 2rem;
    }
    .metric-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 1.5rem;
        border-radius: 12px;
        color: white;
        text-align: center;
        margin-bottom: 1rem;
    }
    .role-card {
        background: #f8f9fa;
        padding: 1rem;
        border-radius: 10px;
        border-left: 4px solid #667eea;
        margin-bottom: 0.8rem;
    }
    .gap-card {
        background: #fff3cd;
        padding: 0.8rem;
        border-radius: 8px;
        border-left: 4px solid #ffc107;
        margin-bottom: 0.5rem;
    }
    .strength-card {
        background: #d4edda;
        padding: 0.8rem;
        border-radius: 8px;
        border-left: 4px solid #28a745;
        margin-bottom: 0.5rem;
    }
    .stProgress > div > div > div > div {
        background-color: #667eea;
    }
</style>
""", unsafe_allow_html=True)

# ─── Constants (must match training code exactly) ───
BRANCHES = ['CSE', 'Mechanical', 'Civil', 'Chemical', 'Metallurgy', 'Electrical', 'ECE', 'AI']
STATUSES = ['4th year student', 'Alumni (Graduated)']
TOOLS_LIST = [
    'Machine Learning / Deep Learning',
    'Data Analysis (Python / Excel / Pandas)',
    'Data Visualization (PowerBI / Tableau)',
    'CAD tools (SolidWorks / CATIA / AutoCAD)',
    'ANSYS / Simulation tools',
    'MATLAB / Simulink',
    'Embedded systems / microcontrollers',
    'Circuit design tools'
]
PROJECT_COUNTS = ['0', '1-2', '3-4', '5+']
PROJECT_DOMAINS = ['Software Development', 'Data Science / AI', 'Core Engineering',
                   'Robotics / Embedded Systems', 'Mixed domains']
INTERNSHIPS = ['No internship', 'Software Development Internship',
               'Data Science / AI Internship', 'Core Engineering Internship',
               'Electronics / Embedded Internship']
PREP_DOMAINS = ['Software Development', 'Data Science / AI', 'Core Engineering',
                'Embedded Systems / Electronics', 'Consulting / Management']

# ─── Skill Requirements for Gap Analysis ───
SKILL_REQUIREMENTS = {
    'Software Developer': {
        'Python_Proficiency': 4, 'CPP_Proficiency': 4, 'Java_Proficiency': 4,
        'DSA_Understanding': 4, 'OOP_Understanding': 4, 'OS_Understanding': 3,
        'Database_SQL_Understanding': 3
    },
    'Data Analyst': {
        'Python_Proficiency': 4, 'Database_SQL_Understanding': 4,
        'DSA_Understanding': 3, 'OOP_Understanding': 3
    },
    'Data Scientist': {
        'Python_Proficiency': 5, 'Database_SQL_Understanding': 4,
        'DSA_Understanding': 4, 'MATLAB_Proficiency': 3
    },
    'Machine Learning Engineer': {
        'Python_Proficiency': 5, 'CPP_Proficiency': 3, 'DSA_Understanding': 4,
        'Database_SQL_Understanding': 3, 'OOP_Understanding': 4
    },
    'DevOps Engineer': {
        'Python_Proficiency': 3, 'OS_Understanding': 4, 'Database_SQL_Understanding': 3,
        'OOP_Understanding': 3
    },
    'Embedded Systems Engineer': {
        'CPP_Proficiency': 4, 'MATLAB_Proficiency': 3, 'OS_Understanding': 4,
        'OOP_Understanding': 3
    },
    'Mechanical Design Engineer': {
        'MATLAB_Proficiency': 4, 'Python_Proficiency': 2
    },
    'Manufacturing Engineer': {
        'MATLAB_Proficiency': 3, 'Python_Proficiency': 2
    },
    'Civil Engineer': {
        'MATLAB_Proficiency': 3
    }
}

IMPROVEMENT_TIPS = {
    'Python_Proficiency': "Complete a Python for Data Science course on Coursera. Practice daily on HackerRank.",
    'CPP_Proficiency': "Solve 50+ C++ problems on Codeforces. Study STL and memory management.",
    'Java_Proficiency': "Build a Spring Boot project. Practice OOP design patterns in Java.",
    'MATLAB_Proficiency': "Complete MATLAB Onramp on MathWorks. Work on a simulation project.",
    'DSA_Understanding': "Solve 100+ LeetCode problems (Easy to Medium). Follow Striver's DSA sheet.",
    'Database_SQL_Understanding': "Complete SQL for Data Science on Coursera. Practice on SQLZoo.",
    'OOP_Understanding': "Study SOLID principles. Implement design patterns in your preferred language.",
    'OS_Understanding': "Study OS concepts from Galvin. Practice process scheduling and memory management."
}

SKILL_DISPLAY_NAMES = {
    'Python_Proficiency': 'Python',
    'CPP_Proficiency': 'C/C++',
    'Java_Proficiency': 'Java',
    'MATLAB_Proficiency': 'MATLAB',
    'DSA_Understanding': 'DSA',
    'Database_SQL_Understanding': 'Database / SQL',
    'OOP_Understanding': 'Object-Oriented Programming',
    'OS_Understanding': 'Operating Systems'
}


# ─── Load Model Artifacts ───
@st.cache_resource
def load_model_artifacts():
    """Load all pickled model artifacts."""
    artifacts = {}
    required_files = {
        'model': 'best_model.pkl',
        'scaler': 'scaler.pkl',
        'label_encoders': 'label_encoders.pkl',
        'target_encoder': 'target_encoder.pkl',
        'mlb': 'mlb_tools.pkl',
        'feature_columns': 'feature_columns.pkl'
    }

    for key, filename in required_files.items():
        filepath = os.path.join(os.path.dirname(__file__), filename)
        if not os.path.exists(filepath):
            st.error(f"Missing file: {filename}. Please run model_training.py first!")
            return None
        with open(filepath, 'rb') as f:
            artifacts[key] = pickle.load(f)

    return artifacts


def predict_roles(student_data, artifacts):
    """Predict top 3 job roles and provide analysis."""
    model = artifacts['model']
    scaler = artifacts['scaler']
    label_encoders = artifacts['label_encoders']
    target_encoder = artifacts['target_encoder']
    mlb = artifacts['mlb']
    feature_columns = artifacts['feature_columns']

    # Build feature vector
    features = {}

    # Numeric features
    numeric_keys = ['CGPA', 'Python_Proficiency', 'CPP_Proficiency',
                    'Java_Proficiency', 'MATLAB_Proficiency',
                    'DSA_Understanding', 'Database_SQL_Understanding',
                    'OOP_Understanding', 'OS_Understanding', 'Confidence_Level']
    for key in numeric_keys:
        features[key] = student_data[key]

    # Categorical features
    categorical_cols = ['Engineering_Branch', 'Student_Status', 'Project_Count',
                        'Project_Domain', 'Internship_Experience', 'Preparation_Domain']
    for col in categorical_cols:
        le = label_encoders[col]
        val = student_data[col]
        if val in le.classes_:
            features[f'{col}_Encoded'] = le.transform([val])[0]
        else:
            features[f'{col}_Encoded'] = 0

    # Tools encoding
    tools_list = student_data['Technical_Tools']
    tools_binary = mlb.transform([tools_list])
    tool_col_names = [f'Tool_{t.split("/")[0].split("(")[0].strip().replace(" ", "_")}'
                      for t in mlb.classes_]
    for i, col_name in enumerate(tool_col_names):
        features[col_name] = tools_binary[0][i]

    # Build DataFrame
    input_df = pd.DataFrame([features])
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[feature_columns]

    # Determine if model needs scaling
    model_name = type(model).__name__
    needs_scaling = model_name in ['LogisticRegression', 'SVC', 'KNeighborsClassifier']

    if needs_scaling:
        input_data = scaler.transform(input_df)
    else:
        input_data = input_df.values

    # Predict
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(input_data)[0]
    else:
        probabilities = np.zeros(len(target_encoder.classes_))
        pred = model.predict(input_data)[0]
        probabilities[pred] = 1.0

    # Top 3
    top3_indices = np.argsort(probabilities)[::-1][:3]
    top3_roles = []
    for idx in top3_indices:
        role_name = target_encoder.classes_[idx]
        confidence = probabilities[idx] * 100
        top3_roles.append({'role': role_name, 'confidence': round(confidence, 1)})

    # Readiness Score
    skill_values = [
        student_data['Python_Proficiency'], student_data['CPP_Proficiency'],
        student_data['Java_Proficiency'], student_data['DSA_Understanding'],
        student_data['Database_SQL_Understanding'], student_data['OOP_Understanding'],
        student_data['OS_Understanding']
    ]
    avg_skills = np.mean(skill_values) / 5 * 40
    cgpa_score = (student_data['CGPA'] / 10) * 30
    confidence_score = (student_data['Confidence_Level'] / 5) * 15
    project_bonus = {'0': 0, '1-2': 5, '3-4': 10, '5+': 15}
    proj_score = project_bonus.get(student_data['Project_Count'], 0)
    readiness_score = round(avg_skills + cgpa_score + confidence_score + proj_score, 1)

    # Skill Gap Analysis
    primary_role = top3_roles[0]['role']
    skill_gaps = []
    strengths = []

    if primary_role in SKILL_REQUIREMENTS:
        for skill, required in SKILL_REQUIREMENTS[primary_role].items():
            current = student_data.get(skill, 0)
            display_name = SKILL_DISPLAY_NAMES.get(skill, skill)
            if current < required:
                skill_gaps.append({
                    'skill': display_name,
                    'skill_key': skill,
                    'current': current,
                    'required': required,
                    'gap': required - current,
                    'tip': IMPROVEMENT_TIPS.get(skill, 'Practice more.')
                })
            else:
                strengths.append({
                    'skill': display_name,
                    'current': current,
                    'required': required
                })

    return {
        'top3_roles': top3_roles,
        'readiness_score': readiness_score,
        'skill_gaps': skill_gaps,
        'strengths': strengths,
        'primary_role': primary_role,
        'all_probabilities': {target_encoder.classes_[i]: round(probabilities[i]*100, 1)
                              for i in range(len(probabilities))}
    }


# ─── Main App ───
def main():
    # Header
    st.markdown('<div class="main-header">Placement Readiness & Job Role Predictor</div>',
                unsafe_allow_html=True)
    st.markdown('<div class="sub-header">AI-powered career guidance for engineering students</div>',
                unsafe_allow_html=True)
    st.markdown("---")

    # Load artifacts
    artifacts = load_model_artifacts()
    if artifacts is None:
        st.stop()

    # Sidebar info
    with st.sidebar:
        st.image("https://img.icons8.com/fluency/96/graduation-cap.png", width=80)
        st.title("About")
        st.markdown("""
        This tool predicts your **top 3 job roles** based on your academic profile,
        skills, and experience. It also provides:

        - **Readiness Score** (0-100)
        - **Skill Gap Analysis**
        - **Personalized Improvement Tips**

        ---
        **Course:** Applied ML (22MET921)
        **Instructor:** Dr. Gunjan Soni
        **Team:** Swapnil, Aditya, Amit
        """)

        model_name = type(artifacts['model']).__name__
        st.info(f"Model: {model_name}")

    # ─── Input Form ───
    st.header("Enter Your Profile")

    col1, col2, col3 = st.columns(3)

    with col1:
        branch = st.selectbox("1. Engineering Branch", BRANCHES)
        status = st.selectbox("2. Your Status", STATUSES)
        cgpa = st.slider("3. CGPA", 4.0, 10.0, 7.5, 0.1)

    with col2:
        st.markdown("**4. Programming Proficiency (1-5)**")
        python_prof = st.slider("Python", 1, 5, 3, key="python")
        cpp_prof = st.slider("C/C++", 1, 5, 3, key="cpp")
        java_prof = st.slider("Java", 1, 5, 2, key="java")
        matlab_prof = st.slider("MATLAB", 1, 5, 2, key="matlab")

    with col3:
        st.markdown("**5. Conceptual Understanding (1-5)**")
        dsa = st.slider("DSA", 1, 5, 3, key="dsa")
        sql = st.slider("Database / SQL", 1, 5, 3, key="sql")
        oop = st.slider("Object-Oriented Programming", 1, 5, 3, key="oop")
        os_und = st.slider("Operating Systems", 1, 5, 3, key="os")

    st.markdown("---")

    col4, col5 = st.columns(2)

    with col4:
        tools = st.multiselect("6. Technical Tools (select all that apply)", TOOLS_LIST)
        project_count = st.selectbox("7. Number of Projects", PROJECT_COUNTS, index=2)
        project_domain = st.selectbox("8. Project Domain", PROJECT_DOMAINS)

    with col5:
        internship = st.selectbox("9. Internship Experience", INTERNSHIPS)
        prep_domain = st.selectbox("10. Preparation Domain", PREP_DOMAINS)
        confidence = st.slider("11. Confidence Level (1-5)", 1, 5, 3)

    st.markdown("---")

    # ─── Predict Button ───
    predict_clicked = st.button("Predict My Job Roles", type="primary", use_container_width=True)

    if predict_clicked:
        if not tools:
            st.warning("Please select at least one technical tool.")
            st.stop()

        # Build student data
        student_data = {
            'Engineering_Branch': branch,
            'Student_Status': status,
            'CGPA': cgpa,
            'Python_Proficiency': python_prof,
            'CPP_Proficiency': cpp_prof,
            'Java_Proficiency': java_prof,
            'MATLAB_Proficiency': matlab_prof,
            'DSA_Understanding': dsa,
            'Database_SQL_Understanding': sql,
            'OOP_Understanding': oop,
            'OS_Understanding': os_und,
            'Technical_Tools': tools,
            'Project_Count': project_count,
            'Project_Domain': project_domain,
            'Internship_Experience': internship,
            'Preparation_Domain': prep_domain,
            'Confidence_Level': confidence
        }

        # Get predictions
        with st.spinner("Analyzing your profile..."):
            result = predict_roles(student_data, artifacts)

        st.markdown("---")
        st.header("Your Placement Readiness Report")

        # ─── Readiness Score ───
        score = result['readiness_score']

        col_score, col_rating = st.columns([1, 2])
        with col_score:
            st.metric("Readiness Score", f"{score}/100")
            st.progress(min(score / 100, 1.0))

        with col_rating:
            if score >= 75:
                st.success("**Excellent!** You are well prepared for placements!")
            elif score >= 60:
                st.info("**Good!** Minor improvements needed in a few areas.")
            elif score >= 45:
                st.warning("**Average.** Focus on the skill gaps identified below.")
            else:
                st.error("**Needs Improvement.** Significant preparation required.")

        st.markdown("---")

        # ─── Top 3 Job Roles ───
        st.subheader("Top 3 Predicted Job Roles")

        for i, role in enumerate(result['top3_roles']):
            col_name, col_bar = st.columns([2, 3])
            with col_name:
                medal = ["🥇", "🥈", "🥉"][i]
                st.markdown(f"### {medal} {role['role']}")
            with col_bar:
                st.metric("Match Confidence", f"{role['confidence']}%")
                st.progress(min(role['confidence'] / 100, 1.0))

        st.markdown("---")

        # ─── All Roles Probability Chart ───
        st.subheader("Match Probability Across All Roles")
        prob_df = pd.DataFrame(
            list(result['all_probabilities'].items()),
            columns=['Job Role', 'Probability (%)']
        ).sort_values('Probability (%)', ascending=True)

        st.bar_chart(
            prob_df.set_index('Job Role'),
            horizontal=True,
            height=400
        )

        st.markdown("---")

        # ─── Skill Analysis ───
        col_str, col_gap = st.columns(2)

        with col_str:
            st.subheader("Your Strengths")
            if result['strengths']:
                for s in result['strengths']:
                    st.markdown(
                        f'<div class="strength-card">'
                        f'<strong>{s["skill"]}</strong><br>'
                        f'Your Level: {s["current"]}/5 | Required: {s["required"]}/5'
                        f'</div>',
                        unsafe_allow_html=True
                    )
            else:
                st.info("Complete the form accurately for a detailed strength analysis.")

        with col_gap:
            st.subheader("Skill Gaps to Address")
            if result['skill_gaps']:
                for gap in result['skill_gaps']:
                    st.markdown(
                        f'<div class="gap-card">'
                        f'<strong>{gap["skill"]}</strong> '
                        f'(Current: {gap["current"]}/5, Required: {gap["required"]}/5, '
                        f'Gap: {gap["gap"]})<br>'
                        f'<em>Tip: {gap["tip"]}</em>'
                        f'</div>',
                        unsafe_allow_html=True
                    )
            else:
                st.success("No significant skill gaps detected! You're well aligned.")

        st.markdown("---")

        # ─── Personalized Roadmap ───
        st.subheader("Personalized Improvement Roadmap")
        primary = result['primary_role']
        st.markdown(f"**Target Role: {primary}**")

        if result['skill_gaps']:
            st.markdown("**Priority actions based on your skill gaps:**")
            for idx, gap in enumerate(sorted(result['skill_gaps'], key=lambda x: x['gap'], reverse=True), 1):
                with st.expander(f"Step {idx}: Improve {gap['skill']} (Gap: {gap['gap']} levels)"):
                    st.write(gap['tip'])
                    st.write(f"**Current Level:** {gap['current']}/5")
                    st.write(f"**Target Level:** {gap['required']}/5")
                    weeks = gap['gap'] * 3
                    st.write(f"**Estimated Time:** {weeks}-{weeks+2} weeks of focused practice")
        else:
            st.balloons()
            st.success("You're already well-equipped for your predicted role! "
                      "Focus on building projects and gaining practical experience.")

        # ─── Profile Summary Table ───
        st.markdown("---")
        with st.expander("View Your Complete Profile Summary"):
            profile_data = {
                'Parameter': ['Branch', 'Status', 'CGPA', 'Python', 'C/C++', 'Java',
                             'MATLAB', 'DSA', 'SQL', 'OOP', 'OS', 'Projects',
                             'Project Domain', 'Internship', 'Prep Domain', 'Confidence'],
                'Value': [branch, status, cgpa, python_prof, cpp_prof, java_prof,
                         matlab_prof, dsa, sql, oop, os_und, project_count,
                         project_domain, internship, prep_domain, confidence]
            }
            st.table(pd.DataFrame(profile_data))

    # ─── Footer ───
    st.markdown("---")
    st.markdown(
        "<div style='text-align: center; color: #888; font-size: 0.85rem;'>"
        "Applied Machine Learning (22MET921) | Dr. Gunjan Soni | "
        "Swapnil Acharya, Aditya Verma, Amit Kumar"
        "</div>",
        unsafe_allow_html=True
    )


if __name__ == "__main__":
    main()

Overwriting app.py


In [37]:
# 1. Install and kill old ports
!pip install pyngrok -q
!fuser -k 80/tcp

# 2. Authenticate
from pyngrok import ngrok
ngrok.set_auth_token("3CPTlAfW9nMarFIOuP7l4QgB1YV_2YqkEGesQ8BQLMQNN1tT6")

# 3. Run the app in the background
import os
os.system("streamlit run app.py --server.port 80 &")

# 4. Open the tunnel
url = ngrok.connect(80)
print("🎯 Click here to open your App:", url)

80/tcp:              35501
🎯 Click here to open your App: NgrokTunnel: "https://annuity-chomp-gangway.ngrok-free.dev" -> "http://localhost:80"
